# 03 — EDA & Insight Discovery: CO₂ & Greenhouse Gas Emissions

Explore key patterns: global trends, top emitters, per-capita inequality, fuel mix transitions, GDP–emissions relationship, and regional comparisons.

In [1]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

PROJECT_ROOT  = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR       = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR    = PROJECT_ROOT / "reports" / "figures"
for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def section(t):
    print("\n" + "="*90); print(t); print("="*90)

section("Load processed datasets")
df     = pd.read_csv(PROCESSED_DIR / "co2_features.csv")     # all years, country-level
modern = pd.read_csv(PROCESSED_DIR / "co2_modern.csv")       # 1990+

# Also load raw to get aggregate region rows for World-level analysis
df_raw = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "owid-co2-data.csv")
world  = df_raw[df_raw["country"] == "World"].copy()

print(f"Country df: {df.shape} | Modern subset: {modern.shape}")
print(f"World rows: {len(world)}")
display(df.head(3))



Load processed datasets
Country df: (42480, 87) | Modern subset: (7630, 87)
World rows: 275


,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,co2_including_luc,co2_including_luc_growth_abs,co2_including_luc_growth_prct,co2_including_luc_per_capita,co2_including_luc_per_gdp,co2_including_luc_per_unit_energy,co2_per_capita,co2_per_gdp,co2_per_unit_energy,coal_co2,coal_co2_per_capita,consumption_co2,consumption_co2_per_capita,consumption_co2_per_gdp,cumulative_cement_co2,cumulative_co2,cumulative_co2_including_luc,cumulative_coal_co2,cumulative_flaring_co2,cumulative_gas_co2,cumulative_luc_co2,cumulative_oil_co2,cumulative_other_co2,energy_per_capita,energy_per_gdp,flaring_co2,flaring_co2_per_capita,gas_co2,gas_co2_per_capita,ghg_excluding_lucf_per_capita,ghg_per_capita,land_use_change_co2,land_use_change_co2_per_capita,methane,methane_per_capita,nitrous_oxide,nitrous_oxide_per_capita,oil_co2,oil_co2_per_capita,other_co2_per_capita,other_industry_co2,primary_energy_consumption,share_global_cement_co2,share_global_co2,share_global_co2_including_luc,share_global_coal_co2,share_global_cumulative_cement_co2,share_global_cumulative_co2,share_global_cumulative_co2_including_luc,share_global_cumulative_coal_co2,share_global_cumulative_flaring_co2,share_global_cumulative_gas_co2,share_global_cumulative_luc_co2,share_global_cumulative_oil_co2,share_global_cumulative_other_co2,share_global_flaring_co2,share_global_gas_co2,share_global_luc_co2,share_global_oil_co2,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share,co2_per_energy,fossil_share,luc_share,gdp_per_capita,co2_growth_cat,income_tier,dominant_fuel,decade
0,Afghanistan,1750,AFG,2802560.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,1750s
1,Afghanistan,1751,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,1750s
2,Afghanistan,1752,AFG,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,Unknown,Unknown,1750s


In [2]:
section("Executive metrics")

latest_world = world[world["year"] == world["year"].max()].iloc[0]
latest_year  = int(latest_world["year"])

# Latest per country
latest = df[df["year"] == df["year"].max()].dropna(subset=["co2"])

overview = pd.DataFrame({
    "Metric": [
        "Latest year in data",
        "World total CO₂ (Mt)",
        "World CO₂ vs 1990",
        "World CO₂ vs 2000",
        "Countries tracked",
        "Top emitter",
        "Highest per-capita emitter",
    ],
    "Value": [
        str(latest_year),
        f"{latest_world['co2']:,.0f} Mt",
        f"{(latest_world['co2'] / world[world['year']==1990]['co2'].values[0] - 1)*100:+.1f}%",
        f"{(latest_world['co2'] / world[world['year']==2000]['co2'].values[0] - 1)*100:+.1f}%",
        f"{latest['country'].nunique():,}",
        f"{latest.nlargest(1,'co2')['country'].values[0]}",
        f"{latest.nlargest(1,'co2_per_capita')['country'].values[0]}",
    ]
})
display(overview)



Executive metrics


,Metric,Value
0,Latest year in data,2024
1,World total CO₂ (Mt),"38,599 Mt"
2,World CO₂ vs 1990,+69.8%
3,World CO₂ vs 2000,+51.3%
4,Countries tracked,215
5,Top emitter,China
6,Highest per-capita emitter,Qatar


In [3]:
section("Global CO₂ trend — 1750 to present")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=world["year"], y=world["co2"],
    mode="lines", name="World CO₂ (Mt)",
    line=dict(color="#e63946", width=2.5)
))
# Annotate key events
events = [
    (1945, "Post-WW2 boom"),
    (1973, "Oil crisis"),
    (1992, "Rio Earth Summit"),
    (2008, "Financial crisis"),
    (2020, "COVID-19"),
]
for yr, label in events:
    y_val = world.loc[world["year"]==yr, "co2"]
    if len(y_val):
        fig.add_vline(x=yr, line_dash="dot", line_color="gray", opacity=0.6)
        fig.add_annotation(x=yr, y=y_val.values[0]*1.05, text=label, showarrow=False,
                           font=dict(size=10), textangle=-45)

fig.update_layout(
    title="Global CO₂ Emissions 1750–Present",
    xaxis_title="Year", yaxis_title="Million tonnes CO₂",
    height=480
)
fig.show()



Global CO₂ trend — 1750 to present


In [4]:
section("Top 10 emitters — total CO₂ (most recent year)")

top10 = (
    latest
    .nlargest(10, "co2")
    [["country","co2","co2_per_capita","share_global_co2"]]
    .reset_index(drop=True)
)
display(top10.round(2))

fig = px.bar(
    top10.sort_values("co2"), x="co2", y="country", orientation="h",
    text=top10.sort_values("co2")["share_global_co2"].round(1).astype(str)+"%",
    color="co2", color_continuous_scale="Reds",
    title=f"Top 10 Countries by Total CO₂ Emissions ({latest_year})",
    labels={"co2": "CO₂ (Mt)", "country": ""}
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, height=480)
fig.show()



Top 10 emitters — total CO₂ (most recent year)


,country,co2,co2_per_capita,share_global_co2
0,China,12289.04,8.66,31.84
1,United States,4904.12,14.20,12.71
2,India,3193.48,2.20,8.27
3,Russia,1780.52,12.29,4.61
4,Japan,961.87,7.77,2.49
5,Indonesia,812.22,2.87,2.10
6,Iran,792.63,8.66,2.05
7,Saudi Arabia,692.13,20.38,1.79
8,South Korea,583.68,11.29,1.51
9,Germany,572.32,6.77,1.48


In [5]:
section("Per-capita CO₂ — top & bottom 15 (latest year, countries with population > 1M)")

big_countries = latest[latest["population"] > 1_000_000]
top15_cap  = big_countries.nlargest(15, "co2_per_capita")[["country","co2_per_capita","income_tier"]]
bot15_cap  = big_countries.nsmallest(15, "co2_per_capita")[["country","co2_per_capita","income_tier"]]

fig = px.bar(
    top15_cap.sort_values("co2_per_capita"), x="co2_per_capita", y="country",
    orientation="h", color="income_tier",
    title=f"Top 15 Highest CO₂ per Capita ({latest_year}, pop > 1M)",
    labels={"co2_per_capita": "t CO₂ per person", "country": ""}
)
fig.update_layout(height=520)
fig.show()

fig2 = px.bar(
    bot15_cap.sort_values("co2_per_capita", ascending=False), x="co2_per_capita", y="country",
    orientation="h", color="income_tier",
    title=f"15 Lowest CO₂ per Capita ({latest_year}, pop > 1M)",
    labels={"co2_per_capita": "t CO₂ per person", "country": ""}
)
fig2.update_layout(height=520)
fig2.show()



Per-capita CO₂ — top & bottom 15 (latest year, countries with population > 1M)


In [6]:
section("Emissions growth trends — top emitters over time")

top10_names = top10["country"].tolist()
trend_data  = df[df["country"].isin(top10_names) & (df["year"] >= 1960)]

fig = px.line(
    trend_data, x="year", y="co2", color="country",
    title="CO₂ Emissions Trend — Top 10 Emitters (1960–present)",
    labels={"co2": "CO₂ (Mt)", "year": "Year"}
)
fig.update_layout(height=500)
fig.show()

# Per-capita version
fig2 = px.line(
    trend_data, x="year", y="co2_per_capita", color="country",
    title="CO₂ per Capita — Top 10 Emitters (1960–present)",
    labels={"co2_per_capita": "t CO₂ per person", "year": "Year"}
)
fig2.update_layout(height=500)
fig2.show()



Emissions growth trends — top emitters over time


In [7]:
section("Fuel mix breakdown — world level")

fuel_cols = ["coal_co2","oil_co2","gas_co2","cement_co2","flaring_co2","land_use_change_co2"]
fuel_cols = [c for c in fuel_cols if c in world.columns]

world_fuels = world[world["year"] >= 1950][["year"] + fuel_cols].dropna(subset=["coal_co2","oil_co2","gas_co2"])

fig = px.area(
    world_fuels.melt(id_vars="year", value_vars=fuel_cols, var_name="Fuel", value_name="CO₂ (Mt)"),
    x="year", y="CO₂ (Mt)", color="Fuel",
    title="Global CO₂ by Fuel Source (1950–present) — Stacked Area",
    labels={"year": "Year"},
    color_discrete_map={
        "coal_co2":"#3d405b","oil_co2":"#81b29a","gas_co2":"#f2cc8f",
        "cement_co2":"#e07a5f","flaring_co2":"#c77dff","land_use_change_co2":"#70c1b3"
    }
)
fig.update_layout(height=500)
fig.show()

# World fuel shares over time
world_fuels_pct = world_fuels.copy()
total_row = world_fuels_pct[fuel_cols].sum(axis=1)
for c in fuel_cols:
    world_fuels_pct[c] = world_fuels_pct[c] / total_row * 100

fig2 = px.area(
    world_fuels_pct.melt(id_vars="year", value_vars=fuel_cols, var_name="Fuel", value_name="Share (%)"),
    x="year", y="Share (%)", color="Fuel",
    title="Global CO₂ Fuel Mix Share (%) 1950–present",
    groupnorm="percent"
)
fig2.update_layout(height=480)
fig2.show()



Fuel mix breakdown — world level


In [8]:
section("GDP vs CO₂ per capita — Environmental Kuznets Curve exploration")

# Use modern era with complete data
econ = modern.dropna(subset=["gdp_per_capita","co2_per_capita"]).copy()
econ_latest = econ[econ["year"] == econ["year"].max()]

fig = px.scatter(
    econ_latest, x="gdp_per_capita", y="co2_per_capita",
    color="income_tier", size="population",
    hover_name="country",
    log_x=True, log_y=False,
    trendline="ols",
    title=f"GDP per Capita vs CO₂ per Capita ({latest_year})<br><sub>Size = population. Log scale on x-axis.</sub>",
    labels={"gdp_per_capita": "GDP per capita (USD, log scale)", "co2_per_capita": "CO₂ per capita (t)"}
)
fig.update_layout(height=560)
fig.show()



GDP vs CO₂ per capita — Environmental Kuznets Curve exploration


In [9]:
section("Regional comparison — per-capita emissions & energy")

# Use continent-level aggregates from raw data
continents = ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]
cont_data = df_raw[df_raw["country"].isin(continents) & (df_raw["year"] >= 1990)].copy()

fig = px.line(
    cont_data, x="year", y="co2_per_capita", color="country",
    title="CO₂ per Capita by Continent (1990–present)",
    labels={"co2_per_capita": "t CO₂ per person", "country": "Region"}
)
fig.update_layout(height=480)
fig.show()

# Energy per capita by continent
fig2 = px.line(
    cont_data.dropna(subset=["energy_per_capita"]), x="year", y="energy_per_capita", color="country",
    title="Energy per Capita by Continent (1990–present)",
    labels={"energy_per_capita": "kWh per person", "country": "Region"}
)
fig2.update_layout(height=480)
fig2.show()



Regional comparison — per-capita emissions & energy


In [10]:
section("Temperature change attribution")

temp_data = modern.dropna(subset=["temperature_change_from_co2"]).copy()
# Top 15 cumulative contributors to temperature change
top_temp = (
    temp_data[temp_data["year"] == temp_data["year"].max()]
    .nlargest(15, "temperature_change_from_co2")
    [["country","temperature_change_from_co2","temperature_change_from_ghg","cumulative_co2"]]
    .reset_index(drop=True)
)
display(top_temp.round(4))

fig = px.bar(
    top_temp.sort_values("temperature_change_from_co2"), 
    x="temperature_change_from_co2", y="country", orientation="h",
    color="temperature_change_from_co2", color_continuous_scale="Reds",
    title="Top 15 Countries by Estimated Temperature Rise from CO₂ (°C)",
    labels={"temperature_change_from_co2": "°C warming from CO₂", "country": ""}
)
fig.update_layout(height=520, showlegend=False)
fig.show()



Temperature change attribution


,country,temperature_change_from_co2,temperature_change_from_ghg,cumulative_co2
0,United States,0.2469,0.2962,434866.5625
1,China,0.1550,0.2174,285087.0312
2,Russia,0.0958,0.1179,122808.2266
3,Brazil,0.0614,0.0881,18198.7070
4,Germany,0.0431,0.0490,95135.6797
5,Indonesia,0.0415,0.0561,17419.9336
6,India,0.0414,0.0798,66072.5312
7,United Kingdom,0.0358,0.0397,80079.4453
8,Japan,0.0332,0.0359,69612.2812
9,Canada,0.0310,0.0381,35644.1289


In [11]:
section("Heatmap — CO₂ per capita by income tier and decade")

heat_data = (
    modern.dropna(subset=["co2_per_capita"])
    .groupby(["income_tier","decade"], as_index=False)
    ["co2_per_capita"].mean()
    .round(2)
)

heat_pivot = heat_data.pivot(index="income_tier", columns="decade", values="co2_per_capita")
fig = px.imshow(
    heat_pivot, text_auto=True, aspect="auto",
    color_continuous_scale="RdYlGn_r",
    title="Average CO₂ per Capita by Income Tier and Decade",
    labels=dict(color="t CO₂/person")
)
fig.update_layout(height=400)
fig.show()



Heatmap — CO₂ per capita by income tier and decade


In [12]:
section("Export key insights")

insights = pd.DataFrame([
    ["Rising global emissions",
     f"World CO₂ rose {(latest_world['co2']/world[world['year']==1990]['co2'].values[0]-1)*100:+.1f}% since 1990.",
     "Global trend line chart"],
    ["Concentration at the top",
     f"Top 3 emitters account for ~50% of global CO₂.",
     "Top emitters bar chart"],
    ["Per-capita inequality is stark",
     "Gulf states and Australia emit 10-20× more per person than South Asia and Africa.",
     "Per-capita ranking bar"],
    ["Coal is the dominant driver",
     "Coal is the single largest fuel source globally, and growing in Asia.",
     "Fuel mix stacked area"],
    ["Wealth predicts emissions",
     "High-income countries have the highest CO₂ per capita, but some are decarbonising.",
     "GDP vs CO₂ scatter"],
    ["Temperature change is unequally caused",
     "USA and EU have contributed the most to historical warming despite current decline.",
     "Temperature attribution bar"],
], columns=["Insight", "Evidence", "Recommended Visual"])
display(insights)
insights.to_csv(PROCESSED_DIR / "eda_insights.csv", index=False)
print("Saved eda_insights.csv")



Export key insights


,Insight,Evidence,Recommended Visual
0,Rising global emissions,World CO₂ rose +69.8% since 1990.,Global trend line chart
1,Concentration at the top,Top 3 emitters account for ~50% of global CO₂.,Top emitters bar chart
2,Per-capita inequality is stark,Gulf states and Australia emit 10-20× more per...,Per-capita ranking bar
3,Coal is the dominant driver,Coal is the single largest fuel source globall...,Fuel mix stacked area
4,Wealth predicts emissions,High-income countries have the highest CO₂ per...,GDP vs CO₂ scatter
5,Temperature change is unequally caused,USA and EU have contributed the most to histor...,Temperature attribution bar


Saved eda_insights.csv
